# 19장 파인튜닝, 지식증류, RAG 비교

이 장은 PDF 교재의 LLM 활용 전략, RAG, 로컬 LLM, LangChain 응용 흐름을 바탕으로 `파인튜닝`, `지식증류`, `RAG`의 차이를 정리합니다.

LLM을 특정 업무에 맞게 활용하려면 여러 선택지가 있습니다. 문서를 검색해 답변하게 만들 수도 있고, 모델 자체를 추가 학습할 수도 있고, 큰 모델의 지식을 작은 모델에 옮길 수도 있습니다. 이 장에서는 세 방법을 비교하고 어떤 상황에서 어떤 방법을 선택하면 좋은지 설명합니다.

## 이 장에서 다루는 내용

1. LLM 맞춤화 전략 개요
2. RAG 개념 복습
3. 파인튜닝 개념
4. 지식증류 개념
5. 세 방법의 핵심 차이
6. 비용, 데이터, 운영 관점 비교
7. 사용 사례별 선택 기준
8. RAG가 적합한 경우
9. 파인튜닝이 적합한 경우
10. 지식증류가 적합한 경우
11. 조합 전략
12. 간단한 의사결정 함수 예시
13. 실무 적용 시 주의점

이 장은 `LLM/llm.ipynb`의 RAG 실습과 로컬 LLM 활용, `LLM/llm2.ipynb`의 DB 챗봇 실습 이후 어떤 방식으로 LLM 앱을 고도화할지 판단하는 정리 장입니다.


## 19.1 LLM 맞춤화 전략 개요

LLM을 업무에 맞게 사용하는 방법은 크게 세 가지로 나눌 수 있습니다.

| 방법 | 핵심 아이디어 |
|---|---|
| RAG | 외부 문서를 검색해 LLM 답변에 근거로 제공 |
| 파인튜닝 | 모델을 추가 학습해 특정 형식, 말투, 작업 패턴을 익히게 함 |
| 지식증류 | 큰 모델의 출력이나 판단 방식을 작은 모델이 따라 하도록 학습 |

세 방법은 서로 경쟁 관계만은 아닙니다. 실제 프로젝트에서는 RAG와 파인튜닝을 함께 쓰거나, 큰 모델로 데이터를 만들고 작은 모델을 학습시키는 방식처럼 조합해서 사용하기도 합니다.


## 19.2 RAG 개념 복습

RAG는 Retrieval-Augmented Generation의 약자로 검색증강생성을 의미합니다.

기본 흐름은 다음과 같습니다.

```text
사용자 질문
  -> 관련 문서 검색
  -> 검색 결과를 프롬프트에 삽입
  -> LLM 답변 생성
```

RAG는 모델을 다시 학습하지 않고 외부 지식을 연결하는 방식입니다. 문서가 자주 바뀌거나, 회사 내부 문서처럼 모델이 원래 알 수 없는 내용을 답해야 할 때 유용합니다.

예시는 다음과 같습니다.

- 사내 규정 문서 Q/A
- PDF 교재 기반 질의응답
- 제품 매뉴얼 챗봇
- 법령, 공지, 정책 문서 검색 챗봇
- DB 조회 결과를 근거로 설명하는 챗봇


## 19.3 파인튜닝 개념

파인튜닝은 이미 학습된 모델을 특정 데이터로 추가 학습하는 방법입니다.

기본 흐름은 다음과 같습니다.

```text
기초 모델
  -> 작업별 학습 데이터 준비
  -> 추가 학습
  -> 특정 작업에 맞춘 모델 생성
```

파인튜닝은 지식을 새로 넣는 방법이라기보다, 모델의 응답 방식과 작업 패턴을 조정하는 데 특히 유용합니다.

적합한 예시는 다음과 같습니다.

- 항상 특정 JSON 형식으로 답변해야 하는 경우
- 특정 말투나 브랜드 톤을 유지해야 하는 경우
- 반복적인 분류 작업을 더 안정적으로 수행해야 하는 경우
- 특정 도메인의 문장 스타일을 익혀야 하는 경우
- 프롬프트만으로는 원하는 출력 형식이 잘 유지되지 않는 경우


## 19.4 지식증류 개념

지식증류는 큰 모델의 지식이나 응답 방식을 작은 모델에 전달하는 학습 방법입니다.

기본 흐름은 다음과 같습니다.

```text
큰 Teacher 모델
  -> 질문에 대한 고품질 답변 생성
  -> 생성된 답변을 학습 데이터로 사용
  -> 작은 Student 모델 학습
```

지식증류는 큰 모델의 성능을 그대로 복제하는 것은 아니지만, 특정 작업에서 작은 모델이 비슷한 패턴을 따라 하도록 만들 수 있습니다.

적합한 예시는 다음과 같습니다.

- 큰 모델을 쓰기에는 비용이 부담되는 경우
- 응답 속도가 중요한 경우
- 온디바이스 또는 로컬 환경에서 작은 모델을 써야 하는 경우
- 특정 작업에 한정해 작은 모델의 성능을 끌어올리고 싶은 경우
- Teacher 모델로 만든 고품질 데이터가 충분한 경우


## 19.5 세 방법의 핵심 차이

| 구분 | RAG | 파인튜닝 | 지식증류 |
|---|---|---|---|
| 핵심 목적 | 외부 지식 연결 | 모델 행동과 출력 방식 조정 | 큰 모델 능력을 작은 모델에 전달 |
| 모델 학습 필요 | 필요 없음 | 필요함 | 필요함 |
| 최신 정보 반영 | 문서만 바꾸면 쉬움 | 다시 학습 필요 | 다시 데이터 생성과 학습 필요 |
| 데이터 형태 | 문서, PDF, DB, 웹 페이지 | 입력-출력 학습쌍 | Teacher 모델의 출력 데이터 |
| 장점 | 지식 업데이트 쉬움 | 출력 형식과 스타일 안정화 | 비용과 속도 최적화 가능 |
| 단점 | 검색 품질에 의존 | 학습 데이터와 비용 필요 | Teacher 품질과 증류 데이터에 의존 |
| 대표 실습 | FAISS, Chroma, Retriever | 분류기, 형식화 모델 | 작은 모델 학습 |

간단히 말하면, `새로운 지식`을 연결하려면 RAG가 먼저이고, `응답 방식`을 바꾸려면 파인튜닝을 고려하고, `비용과 속도`를 줄이려면 지식증류를 고려합니다.


## 19.6 비용, 데이터, 운영 관점 비교

| 관점 | RAG | 파인튜닝 | 지식증류 |
|---|---|---|---|
| 초기 구축 난이도 | 중간 | 높음 | 높음 |
| 운영 난이도 | 문서 품질과 검색 관리 필요 | 모델 버전 관리 필요 | Teacher/Student 데이터 관리 필요 |
| 데이터 준비 | 원문 문서 중심 | 정제된 입력-출력 쌍 필요 | Teacher가 만든 학습 데이터 필요 |
| 비용 구조 | 검색 인프라 + LLM 호출 비용 | 학습 비용 + 추론 비용 | 데이터 생성 비용 + 학습 비용 + 추론 비용 |
| 변경 대응 | 문서 교체로 빠름 | 재학습 필요 | 재증류 또는 재학습 필요 |
| 설명 가능성 | 근거 문서를 제시하기 쉬움 | 학습 결과라 근거 제시 어려움 | 증류 데이터 추적 필요 |

실무에서는 처음부터 파인튜닝을 선택하기보다, 먼저 프롬프트와 RAG로 해결 가능한지 확인하는 경우가 많습니다. 파인튜닝은 데이터 준비와 평가 체계가 갖춰진 뒤 적용하는 것이 안정적입니다.


In [ ]:
# 교재 위치: 19장 파인튜닝, 지식증류, RAG 비교 - 비교 표 데이터 만들기
# 이 셀은 RAG, 파인튜닝, 지식증류의 차이를 pandas 표로 정리합니다.

# pandas는 비교 표를 DataFrame 형태로 만들기 위해 사용합니다.
import pandas as pd

# comparison_data 변수에는 비교 항목별 내용을 리스트로 저장합니다.
comparison_data = [
    # RAG에 대한 비교 정보입니다.
    {"method": "RAG", "main_goal": "외부 지식 연결", "training": "불필요", "best_for": "문서 기반 질의응답", "update": "문서 교체로 빠름"},
    # 파인튜닝에 대한 비교 정보입니다.
    {"method": "파인튜닝", "main_goal": "응답 방식 조정", "training": "필요", "best_for": "형식, 스타일, 반복 작업", "update": "재학습 필요"},
    # 지식증류에 대한 비교 정보입니다.
    {"method": "지식증류", "main_goal": "작은 모델 최적화", "training": "필요", "best_for": "속도와 비용 최적화", "update": "재증류 필요"},
]

# comparison_df는 비교 정보를 표 형태로 담은 DataFrame입니다.
comparison_df = pd.DataFrame(comparison_data)

# 비교 표를 출력합니다.
comparison_df


## 19.7 RAG가 적합한 경우

RAG는 다음 조건에서 특히 적합합니다.

- 답변 근거가 문서에 있어야 합니다.
- 문서가 자주 업데이트됩니다.
- 모델이 원래 모르는 내부 자료를 사용해야 합니다.
- 답변과 함께 출처나 근거를 보여줘야 합니다.
- 모델을 학습할 만큼의 정제된 데이터가 없습니다.
- 빠르게 프로토타입을 만들어야 합니다.

예를 들어 PDF 교재 기반 챗봇, 회사 규정 챗봇, 제품 매뉴얼 챗봇은 RAG가 좋은 출발점입니다.

RAG의 품질은 다음 요소에 영향을 받습니다.

- 문서 전처리 품질
- 청크 크기와 overlap 설정
- 임베딩 모델 품질
- 벡터스토어 검색 설정
- 프롬프트에서 근거를 사용하는 방식
- 검색 결과가 없을 때의 처리 방식


## 19.8 파인튜닝이 적합한 경우

파인튜닝은 다음 조건에서 고려할 수 있습니다.

- 동일한 유형의 작업을 매우 많이 반복합니다.
- 출력 형식이 항상 일정해야 합니다.
- 프롬프트만으로 원하는 스타일이 안정적으로 나오지 않습니다.
- 특정 도메인의 문체나 라벨링 기준을 모델이 익혀야 합니다.
- 충분한 입력-출력 예시 데이터가 있습니다.
- 모델 평가 기준과 테스트 데이터셋이 준비되어 있습니다.

예시는 다음과 같습니다.

- 고객 문의를 정해진 카테고리로 분류
- 상담 내용을 특정 JSON 형식으로 변환
- 회사 브랜드 톤에 맞춘 짧은 문구 생성
- 의료, 법률, 금융처럼 표현 형식이 엄격한 문서 보조

파인튜닝은 모델에 모든 지식을 넣는 만능 방법이 아닙니다. 최신 정보나 긴 문서 지식은 RAG로 연결하는 편이 더 적합한 경우가 많습니다.


## 19.9 지식증류가 적합한 경우

지식증류는 큰 모델을 계속 사용하기 어렵거나, 작은 모델을 특정 작업에 맞게 최적화하고 싶을 때 적합합니다.

다음 조건에서 고려할 수 있습니다.

- 큰 모델 호출 비용이 높습니다.
- 응답 속도가 매우 중요합니다.
- 로컬 또는 제한된 환경에서 실행해야 합니다.
- 특정 작업 범위가 비교적 명확합니다.
- Teacher 모델로 고품질 학습 데이터를 만들 수 있습니다.

예시는 다음과 같습니다.

- 큰 모델이 생성한 분류 라벨을 작은 모델이 학습
- 큰 모델의 답변 스타일을 작은 모델에 학습
- 특정 FAQ 도메인에서 작은 챗봇 모델 만들기
- 모바일이나 엣지 환경에서 빠르게 동작하는 모델 만들기

지식증류에서는 Teacher 모델의 오류도 Student 모델에 전달될 수 있으므로, 학습 데이터 품질 검토가 중요합니다.


## 19.10 조합 전략

실제 LLM 시스템은 하나의 방법만 쓰지 않을 수 있습니다. 대표적인 조합은 다음과 같습니다.

| 조합 | 설명 |
|---|---|
| RAG + 프롬프트 엔지니어링 | 가장 기본적인 지식 기반 챗봇 구성 |
| RAG + 파인튜닝 | 문서는 검색하고, 답변 형식과 스타일은 학습으로 안정화 |
| Teacher 모델 + 지식증류 | 큰 모델로 데이터 생성 후 작은 모델 학습 |
| RAG + LangGraph | 검색, 검증, 답변, 오류 처리를 그래프로 관리 |
| RAG + LangSmith | 검색 품질과 답변 품질을 추적하고 평가 |
| DB 챗봇 + RAG | 정형 데이터는 SQL로 조회하고 비정형 문서는 RAG로 검색 |

예를 들어 사내 문서 챗봇을 만든다면 처음에는 RAG로 시작하고, 이후 답변 형식이 계속 흔들리면 파인튜닝을 검토할 수 있습니다. 사용자가 많아져 비용이 커지면 지식증류나 작은 모델 사용을 검토할 수 있습니다.


## 19.11 선택 기준 정리

다음 질문에 답해 보면 어떤 전략이 적합한지 판단하기 쉽습니다.

| 질문 | 추천 방향 |
|---|---|
| 최신 문서나 내부 자료를 답변해야 하는가? | RAG |
| 답변 근거를 보여줘야 하는가? | RAG |
| 출력 형식이 자주 깨지는가? | 파인튜닝 또는 강한 프롬프트 |
| 같은 작업을 대량으로 반복하는가? | 파인튜닝 |
| 큰 모델 비용이 부담되는가? | 지식증류 또는 작은 모델 |
| 빠른 로컬 실행이 필요한가? | 지식증류 또는 로컬 소형 모델 |
| 데이터가 아직 부족한가? | RAG와 프롬프트부터 시작 |
| 평가 데이터셋이 준비되어 있는가? | 파인튜닝 또는 지식증류 검토 가능 |


In [ ]:
# 교재 위치: 19장 파인튜닝, 지식증류, RAG 비교 - 선택 기준 함수
# 이 셀은 요구사항에 따라 추천 전략을 간단히 반환하는 예시 함수입니다.

# recommend_strategy 함수는 프로젝트 조건을 받아 추천 전략을 반환합니다.
def recommend_strategy(needs_latest_knowledge: bool, needs_fixed_format: bool, has_training_data: bool, needs_low_cost: bool) -> str:
    # 최신 문서나 내부 지식이 중요하면 RAG를 우선 추천합니다.
    if needs_latest_knowledge:
        # RAG는 외부 문서를 바꾸는 방식으로 지식을 빠르게 갱신할 수 있습니다.
        return "RAG를 우선 고려하세요. 문서 검색과 근거 기반 답변이 핵심입니다."

    # 정해진 출력 형식이 중요하고 학습 데이터가 있으면 파인튜닝을 추천합니다.
    if needs_fixed_format and has_training_data:
        # 파인튜닝은 반복적인 형식과 스타일을 안정화하는 데 유리합니다.
        return "파인튜닝을 고려하세요. 출력 형식과 작업 패턴을 안정화할 수 있습니다."

    # 비용과 속도가 중요하고 학습 데이터를 만들 수 있으면 지식증류를 추천합니다.
    if needs_low_cost and has_training_data:
        # 지식증류는 작은 모델로 비용과 속도를 최적화하는 데 사용할 수 있습니다.
        return "지식증류를 고려하세요. 작은 모델로 비용과 응답 속도를 개선할 수 있습니다."

    # 학습 데이터가 부족하면 프롬프트와 RAG부터 시작하는 것이 일반적입니다.
    if not has_training_data:
        # 데이터가 부족한 초기 단계에서는 학습보다 프롬프트와 검색 구성이 현실적입니다.
        return "프롬프트 엔지니어링과 RAG부터 시작하세요. 학습 데이터가 쌓이면 파인튜닝을 검토하세요."

    # 특별한 조건이 없으면 단순한 프롬프트 기반 체인부터 시작하도록 안내합니다.
    return "먼저 프롬프트 기반 체인으로 구현한 뒤, 문제점에 따라 RAG, 파인튜닝, 지식증류를 선택하세요."


# 최신 내부 문서가 필요한 프로젝트 예시입니다.
example_recommendation_1 = recommend_strategy(True, False, False, False)

# 고정 JSON 형식과 학습 데이터가 있는 프로젝트 예시입니다.
example_recommendation_2 = recommend_strategy(False, True, True, False)

# 비용 절감이 중요하고 학습 데이터가 있는 프로젝트 예시입니다.
example_recommendation_3 = recommend_strategy(False, False, True, True)

# 추천 결과를 출력합니다.
print(example_recommendation_1)

# 두 번째 추천 결과를 출력합니다.
print(example_recommendation_2)

# 세 번째 추천 결과를 출력합니다.
print(example_recommendation_3)


## 19.12 간단한 Gradio 의사결정 도우미

Gradio를 사용하면 프로젝트 조건을 체크하고 추천 전략을 보여주는 간단한 도구를 만들 수 있습니다.

이 도구는 실제 컨설팅을 대신하는 것은 아니지만, 학습자가 세 전략의 차이를 이해하는 데 도움이 됩니다.


In [ ]:
# 교재 위치: 19장 파인튜닝, 지식증류, RAG 비교 - Gradio 의사결정 도우미
# 이 셀은 RAG, 파인튜닝, 지식증류 중 어떤 전략을 고려할지 추천하는 Gradio UI를 만듭니다.

# gradio는 웹 UI를 만들기 위해 사용합니다.
import gradio as gr


# recommend_from_ui 함수는 Gradio 체크박스 입력을 recommend_strategy 함수로 전달합니다.
def recommend_from_ui(needs_latest_knowledge: bool, needs_fixed_format: bool, has_training_data: bool, needs_low_cost: bool) -> str:
    # recommend_strategy 함수에 UI 입력값을 전달해 추천 문장을 생성합니다.
    recommendation = recommend_strategy(needs_latest_knowledge, needs_fixed_format, has_training_data, needs_low_cost)

    # 추천 문장을 반환합니다.
    return recommendation


# Blocks는 여러 Gradio 컴포넌트를 배치할 수 있는 컨테이너입니다.
with gr.Blocks() as strategy_demo:
    # Markdown은 화면 제목을 표시합니다.
    gr.Markdown("# LLM 맞춤화 전략 선택 도우미")

    # 최신 지식 필요 여부를 입력받는 Checkbox입니다.
    latest_checkbox = gr.Checkbox(label="최신 문서나 내부 자료를 답변해야 한다")

    # 고정 출력 형식 필요 여부를 입력받는 Checkbox입니다.
    format_checkbox = gr.Checkbox(label="항상 고정된 출력 형식이 필요하다")

    # 학습 데이터 보유 여부를 입력받는 Checkbox입니다.
    data_checkbox = gr.Checkbox(label="정제된 학습 데이터가 있다")

    # 비용 절감 필요 여부를 입력받는 Checkbox입니다.
    cost_checkbox = gr.Checkbox(label="큰 모델 비용이나 응답 속도가 부담된다")

    # 추천 실행 버튼입니다.
    recommend_button = gr.Button("전략 추천하기")

    # 추천 결과를 표시할 Textbox입니다.
    recommendation_output = gr.Textbox(label="추천 전략", lines=5)

    # 버튼 클릭 시 recommend_from_ui 함수를 실행합니다.
    recommend_button.click(
        # fn에는 실행할 추천 함수를 지정합니다.
        fn=recommend_from_ui,
        # inputs에는 네 개의 체크박스를 지정합니다.
        inputs=[latest_checkbox, format_checkbox, data_checkbox, cost_checkbox],
        # outputs에는 추천 결과 Textbox를 지정합니다.
        outputs=recommendation_output,
    )

# launch를 실행하면 로컬 웹 브라우저에서 전략 선택 도우미를 확인할 수 있습니다.
# strategy_demo.launch()


## 19.13 실무 적용 시 주의점

RAG, 파인튜닝, 지식증류를 선택할 때는 기술적 가능성뿐 아니라 운영 조건을 함께 봐야 합니다.

주의할 점은 다음과 같습니다.

- RAG는 문서 품질이 낮으면 답변 품질도 낮아집니다.
- 파인튜닝은 잘못된 데이터로 학습하면 오류 패턴도 함께 학습합니다.
- 지식증류는 Teacher 모델의 편향과 오류가 Student 모델로 전달될 수 있습니다.
- 세 방법 모두 평가 데이터셋이 없으면 개선 여부를 판단하기 어렵습니다.
- 개인정보와 저작권이 포함된 데이터는 학습이나 검색 인덱싱 전에 검토해야 합니다.
- 모델을 바꾸면 프롬프트, 검색 결과, 응답 품질이 모두 달라질 수 있습니다.
- 운영 환경에서는 LangSmith 같은 추적 도구로 실패 사례를 모으는 것이 좋습니다.

가장 현실적인 시작점은 작은 범위의 RAG 또는 프롬프트 기반 프로토타입을 만들고, 실패 사례를 모아 개선 방향을 정하는 것입니다.


## 19.14 정리

이 장에서는 RAG, 파인튜닝, 지식증류의 차이와 선택 기준을 정리했습니다.

핵심 정리는 다음과 같습니다.

- RAG는 외부 문서를 검색해 LLM 답변에 근거로 제공하는 방식입니다.
- 파인튜닝은 모델을 추가 학습해 특정 출력 형식, 스타일, 작업 패턴을 익히게 하는 방식입니다.
- 지식증류는 큰 모델의 지식이나 출력 패턴을 작은 모델이 따라 하도록 학습하는 방식입니다.
- 최신 정보와 근거 제시가 중요하면 RAG를 우선 고려합니다.
- 고정 형식과 반복 작업이 중요하고 데이터가 충분하면 파인튜닝을 고려합니다.
- 비용과 응답 속도가 중요하고 학습 데이터가 있으면 지식증류를 고려합니다.
- 실제 프로젝트에서는 RAG, 파인튜닝, 지식증류를 조합해서 사용할 수 있습니다.
- 어떤 전략을 선택하든 평가 데이터와 실패 사례 관리가 중요합니다.

다음 장에서는 다중 에이전트와 AutoGen을 다룹니다. 여러 역할을 가진 에이전트가 협업하는 구조와 LLM 기반 자동화 흐름을 정리합니다.
